In [1]:
import sys
sys.path.insert(0, '../../gofher')

import os
import matplotlib.image as mpimg

from gofher import run_gofher
from visualize import visualize
from file_helper import write_csv,check_if_folder_exists_and_create
from spin_parity import read_spin_parity_galaxies_label_from_csv, standardize_galaxy_name

In [2]:

#survery_to_use = "panstarrs"

#BANDS_IN_ORDER = ['g','r','i','z','y'] #Important: Must stay in order of BLUEST to REDDEST Waveband (Editting this will cause gofher to no longer correctly evaluate redder side of galaxy)
#REF_BANDS_IN_ORDER = ['i','z','y','r','g'] #The prefernce each waveband being choosen as refernce band from highest priority to lowest priority

#'''
survery_to_use = "sdss" #Note: for sdss we are not using u do to poor quality

BANDS_IN_ORDER = ['g','r','i','z'] #Important: Must stay in order of BLUEST to REDDEST Waveband (Editting this will cause gofher to no longer correctly evaluate redder side of galaxy)
REF_BANDS_IN_ORDER = ['r','i','z','g'] #The prefernce each waveband being choosen as refernce band from highest priority to lowest priority
#'''

In [3]:
figure_to_run_on = "figure11"
#panstarrs:
bin_size = None #None or a positive integer

#sdss:
#NOTE: sdss needs to flip color image
##bin_size = 4 #should be 4
#Source: "The median seeing of all SDSS imaging data (using the psfWidth metric) is 1.32 arcseconds in the r-band."
#"The pixel size in the Sloan Digital Sky Survey (SDSS) is 0.396 arcseconds per pixel" - https://classic.sdss.org/dr3/instruments/imager/
bin_prior_to_param_fitting = True

In [4]:
generate_verbose_csv = True
generate_ebm_csv = False
generate_params_csv = True
generate_visualization = True
save_visualization = True

In [5]:
#Important: Make sure you update these values:
path_to_catalog_data = "..\\..\\..\\spin-parity-catalog-data"
#path_to_output = "..\\..\\..\\gofher-data\\panstarrs\\default_ellipse_mask_fitting" #- https://www.sdss4.org/dr17/imaging/other_info/
path_to_output = "C:\\Users\\school\\Desktop\\gofher_was_p_val"
#path_to_output = "E:\\grad_school\\research\\winter_2025\\mannwhitneyu_sdss"

In [6]:
def get_fits_path(name,band):
    """the file path of where existing fits files can be found"""
    return os.path.join(path_to_catalog_data,survery_to_use,figure_to_run_on,name,"{}_{}.fits".format(name,band))

def get_color_image_path(name):
    file_type = "png"
    if survery_to_use == "panstarrs": file_type = "jfif"
    return os.path.join(path_to_catalog_data,survery_to_use,figure_to_run_on,name,"{}_color.{}".format(name,file_type))

def get_path_to_catalog_csv():
    return os.path.join(path_to_catalog_data,"catalog","{}.csv".format(figure_to_run_on))

In [7]:
def get_paper_dark_side_labels():
    return read_spin_parity_galaxies_label_from_csv(get_path_to_catalog_csv())

def get_galaxies():
    return os.listdir(os.path.join(path_to_catalog_data,survery_to_use,figure_to_run_on))

In [ ]:
def run_gofher_on_catalog():
    paper_labels = get_paper_dark_side_labels()

    verbose_header = []
    verbose_rows = []

    ebm_header = []
    ebm_rows = []

    params_header = []
    params_rows = []

    i = 1


    for name in get_galaxies():
        #if name != "NGC3169": continue

        if standardize_galaxy_name(name) not in paper_labels:
            print("skippimg",name)
            continue

        #if name.lower() != "ngc3368": continue

        print(name, i,"of",len(get_galaxies()))

        try:
            paper_label = paper_labels[standardize_galaxy_name(name)]
            gal = run_gofher(name,get_fits_path,BANDS_IN_ORDER,REF_BANDS_IN_ORDER, paper_label,s=bin_size,bin_prior_to_param_fitting=bin_prior_to_param_fitting)

            if generate_visualization:
                save_path = ''
                
                if save_visualization:
                    sub_folder = os.path.join(path_to_output,figure_to_run_on)
                    check_if_folder_exists_and_create(sub_folder)
                    save_path = os.path.join(sub_folder,"{}.png".format(name))

                color_image = color = mpimg.imread(get_color_image_path(name))
                visualize(gal,color_image,BANDS_IN_ORDER,paper_label,save_path=save_path,color_flip=(survery_to_use=="sdss"),show_stats=False)

            if generate_verbose_csv:
                (header,row) = gal.get_verbose_csv_header_and_row(BANDS_IN_ORDER,paper_label)
                if len(verbose_header) == 0: verbose_header = header
                verbose_rows.append(row)

            if generate_ebm_csv:
                (header,row) = gal.get_ebm_csv_header_and_row(BANDS_IN_ORDER, paper_label)
                if len(ebm_header) == 0: ebm_header = header
                ebm_rows.append(row)

            if generate_params_csv:
                (header,row) = gal.get_params_csv_header_and_row()
                if len(params_header) == 0: params_header = header
                params_rows.append(row)

        except Exception as e:
            print(e)
        i += 1
        #break

    if generate_verbose_csv:
        verbose_csv_path = os.path.join(path_to_output,"{}_verbose.csv".format(figure_to_run_on))
        write_csv(verbose_csv_path,verbose_header,verbose_rows)

    if generate_ebm_csv:
        ebm_csv_path = os.path.join(path_to_output,"{}_ebm.csv".format(figure_to_run_on))
        write_csv(ebm_csv_path,ebm_header,ebm_rows)

    if generate_params_csv:
        params_csv_path = os.path.join(path_to_output,"{}_params.csv".format(figure_to_run_on))
        write_csv(params_csv_path,params_header,params_rows)

In [9]:
if not os.path.exists(path_to_catalog_data):
    raise ValueError("The path to the catalog is not found {} - make sure you update path_to_catalog_data".format(path_to_catalog_data))

if not os.path.exists(path_to_output):
    raise ValueError("The path output is not found {} - make sure you update ppath_to_output".format(path_to_catalog_data))


run_gofher_on_catalog()

IC2247 1 of 25
g-r
-0.014723475540765245 0.1361121834719221
0.0125 0.16333565901268735
-0.1240641910028385 0.0853897543472597
0.012500000000000011 0.22195394535009821
g-i
-0.013991036564462834 0.24732473885394934
0.012499999999999999 0.2738157754184122
-0.18010280903120446 0.181798026880795
0.012500000000000011 0.37440083591199946
g-z
-0.017117618368050234 0.3055547707100269
0.0125 0.3351723890780771
-0.2660332868943416 0.24574416593649512
0.012500000000000011 0.5242774528308367
r-i
-0.004241970907562259 0.12443498822662746
0.0125 0.14117695913418973
-0.0754073175871981 0.09640827253353529
0.012499999999999997 0.1843155901207334
r-z
-0.011559802276233839 0.20520505830919633
0.0125 0.22926486058543016
-0.16133779545033522 0.1626837396949784
0.012500000000000011 0.33652153514531363
i-z
-0.00946108193363528 0.08889354451699383
0.0125 0.1108546264506291
-0.0987211727249604 0.07500097327483224
0.012499999999999997 0.18622214599979264
